In [1]:
import re
import pickle
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel

/home/indra/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
dataPath = "../data/Beauty/metadata.json"
rows = []
with open(dataPath) as f:
    for line in f:
        rows.append(eval(line))
df = pd.DataFrame(rows)
df.head()

,asin,description,title,imUrl,salesRank,categories,price,related,brand
0,0205616461,"As we age, our once youthful, healthy skin suc...",Bio-Active Anti-Aging Serum (Firming Ultra-Hyd...,http://ecx.images-amazon.com/images/I/41DecrGO...,{'Health & Personal Care': 461765},"[[Beauty, Skin Care, Face, Creams & Moisturize...",NaN,NaN,NaN
1,0558925278,Mineral Powder Brush--Apply powder or mineral ...,Eco Friendly Ecotools Quality Natural Bamboo C...,http://ecx.images-amazon.com/images/I/51L%2BzY...,{'Beauty': 402875},"[[Beauty, Tools & Accessories, Makeup Brushes ...",NaN,NaN,NaN
2,0733001998,"From the Greek island of Chios, this Mastiha b...",Mastiha Body Lotion,http://ecx.images-amazon.com/images/I/311WK5y1...,{'Beauty': 540255},"[[Beauty, Skin Care, Body, Moisturizers, Lotio...",NaN,NaN,NaN
3,0737104473,Limited edition Hello Kitty Lipstick featuring...,Hello Kitty Lustre Lipstick (See sellers comme...,http://ecx.images-amazon.com/images/I/31u6Hrzk...,{'Beauty': 931125},"[[Beauty, Makeup, Lips, Lipstick]]",NaN,NaN,NaN
4,0762451459,"The mermaid is an elusive (okay, mythical) cre...",Stephanie Johnson Mermaid Round Snap Mirror,http://ecx.images-amazon.com/images/I/41y2%2BF...,NaN,"[[Beauty, Tools & Accessories, Mirrors, Makeup...",19.98,NaN,NaN


In [10]:
movies_path = "../data/ml-1m/movies.dat"
output_path  = "../data/ml-1m/content_embeddings.pkl"
model_name   = "meta-llama/Llama-2-7b-hf"

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 32
MAX_LENGTH = 40

df = pd.read_csv(
    movies_path,
    sep="::",
    engine="python",
    header=None,
    encoding="latin-1",
    names=["movie_id", "title", "genres"]
)
print(f"{len(df)} movies loaded")
df.head()

3883 movies loaded


,movie_id,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


In [11]:
def build_text(row):
    title = row["title"]
    year_match = re.search(r"\((\d{4})\)", title)
    year = year_match.group(1) if year_match else ""
    name = re.sub(r"\(\d{4}\)", "", title).strip()
    genres = row["genres"].replace("|", " ")
    return f"{name} {year} {genres}".strip()

df["text"] = df.apply(build_text, axis=1)
df[["movie_id", "text"]].head(10)

,movie_id,text
0,1,Toy Story 1995 Animation Children's Comedy
1,2,Jumanji 1995 Adventure Children's Fantasy
2,3,Grumpier Old Men 1995 Comedy Romance
3,4,Waiting to Exhale 1995 Comedy Drama
4,5,Father of the Bride Part II 1995 Comedy
5,6,Heat 1995 Action Crime Thriller
6,7,Sabrina 1995 Comedy Romance
7,8,Tom and Huck 1995 Adventure Children's
8,9,Sudden Death 1995 Action
9,10,GoldenEye 1995 Action Adventure Thriller


In [12]:
# Requires HuggingFace access — run `huggingface-cli login` in terminal if not logged in
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModel.from_pretrained(model_name, torch_dtype=torch.float16)
model = model.to(DEVICE).eval()
print(f"Loaded {model_name} on {DEVICE}")
print(f"Hidden dim: {model.config.hidden_size}")

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1959.75it/s]
[transformers] LlamaModel LOAD REPORT from: meta-llama/Llama-2-7b-hf
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded meta-llama/Llama-2-7b-hf on cuda
Hidden dim: 4096


In [13]:
def get_embeddings(texts):
    all_embeddings = []
    for i in range(0, len(texts), BATCH_SIZE):
        batch = texts[i : i + BATCH_SIZE]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH
        ).to(DEVICE)
        with torch.no_grad():
            outputs = model(**inputs)
        mask = inputs["attention_mask"].unsqueeze(-1).float()
        embeddings = (outputs.last_hidden_state * mask).sum(1) / mask.sum(1)
        all_embeddings.append(embeddings.cpu().float().numpy())
        if (i // BATCH_SIZE) % 10 == 0:
            print(f"  {i + len(batch)}/{len(texts)} done")
    return np.concatenate(all_embeddings, axis=0)

print("Generating embeddings...")
embeddings = get_embeddings(df["text"].tolist())
print(f"Embeddings shape: {embeddings.shape}")

Generating embeddings...
  32/3883 done
  352/3883 done
  672/3883 done
  992/3883 done
  1312/3883 done
  1632/3883 done
  1952/3883 done
  2272/3883 done
  2592/3883 done
  2912/3883 done
  3232/3883 done
  3552/3883 done
  3872/3883 done
Embeddings shape: (3883, 4096)


In [14]:
data = {
    "item_id":   df["movie_id"].tolist(),
    "embedding": embeddings.tolist()
}

with open(output_path, "wb") as f:
    pickle.dump(data, f)

print(f"Saved {len(data['item_id'])} embeddings → {output_path}")
print(f"Embedding dim: {len(data['embedding'][0])}")

Saved 3883 embeddings → ../data/ml-1m/content_embeddings.pkl
Embedding dim: 4096
